In [1]:
from transformers import AutoImageProcessor, ResNetModel
from PIL import Image
import torch
import torch.nn.functional as F

2025-12-09 14:51:10.070988: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765291870.261133      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765291870.315371      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
class MicrosoftResNet50FeatureExtractor:
    def __init__(self, device="cuda"):
        self.device = device
        self.processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
        self.model = ResNetModel.from_pretrained("microsoft/resnet-50").to(device)
        self.model.eval()

    def feature_extraction(self, img_path):
        img = Image.open(img_path).convert("RGB")
        inputs = self.processor(images=img, return_tensors="pt").to(self.device)

        with torch.no_grad():
            out = self.model(**inputs)

        feat_map = out.last_hidden_state        # [1, 2048, 7, 7]
        gap = F.adaptive_avg_pool2d(feat_map, (1, 1))  # [1, 2048, 1, 1]
        vec = gap.view(gap.size(0), -1)         # [1, 2048]

        return vec  # torch tensor
        
device = "cuda"
extractor = MicrosoftResNet50FeatureExtractor(device)

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [3]:
import os
import numpy as np
from tqdm import tqdm

# 1. Đường dẫn dataset trên Kaggle
DATASET_ROOT = "/kaggle/input/vietnamese-foods/Images"   # sửa lại tên dataset của bạn
OUTPUT_ROOT  = "/kaggle/working/ResNet50-embeddings-vectors"        # nơi lưu vector

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# 3. Hàm xử lý 1 split (Train / Validate / Test)
def process_split(split_name):
    split_input_dir = os.path.join(DATASET_ROOT, split_name)   # vd: /kaggle/input/.../Train
    split_output_dir = os.path.join(OUTPUT_ROOT, split_name)   # vd: /kaggle/working/embeddings/Train

    for class_name in os.listdir(split_input_dir):
        class_input_dir = os.path.join(split_input_dir, class_name)
        if not os.path.isdir(class_input_dir):
            continue

        class_output_dir = os.path.join(split_output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)

        # Lọc các file ảnh
        img_files = [
            f for f in os.listdir(class_input_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

        print(f"[{split_name}] Class: {class_name} - {len(img_files)} images")

        for fname in tqdm(img_files):
            img_path = os.path.join(class_input_dir, fname)

            # 1) Trích xuất feature vector (torch tensor [1, 2048])
            feat = extractor.feature_extraction(img_path)

            # 2) Chuyển sang numpy
            feat_np = feat.squeeze(0).cpu().numpy()   # (2048,)

            # 3) Tên file .npy tương ứng
            base_name = os.path.splitext(fname)[0]    # 001
            out_path = os.path.join(class_output_dir, base_name + ".npy")

            # 4) Lưu
            np.save(out_path, feat_np)


In [4]:
# 4. Chạy cho cả 3 folder
for split in ["Train", "Validate", "Test"]:
    process_split(split)

[Train] Class: Banh cuon - 798 images


100%|██████████| 798/798 [00:30<00:00, 25.99it/s]


[Train] Class: Mi quang - 618 images


100%|██████████| 618/618 [00:22<00:00, 27.61it/s]


[Train] Class: Banh trang nuong - 556 images


...

100%|██████████| 235/235 [00:08<00:00, 27.24it/s]


[Test] Class: Com tam - 189 images


100%|██████████| 189/189 [00:07<00:00, 25.35it/s]


[Test] Class: Hu tieu - 197 images


100%|██████████| 197/197 [00:07<00:00, 27.96it/s]
